# SASRec on MovieLens 1M — Full Rating Signal (1–5)

This notebook is a companion to `sasrec_movielens1m_positives.ipynb` and demonstrates
**full-signal** training using normalized 1–5 ratings as soft BCE labels.

## Key differences from the positives-only notebook

| | Positives-only (notebook 1) | Full-signal (this notebook) |
|---|---|---|
| Interactions used | Rating ≥ 4 only | All 1–5 ratings |
| OUTCOME | 1.0 (binary) | Rating/5 (0.2–1.0, soft label) |
| Estimator | `SASRecClassifierEstimator` (BCE) | `SASRecClassifierEstimator` (BCE, soft labels) |
| Random negatives | target = 0.0 | target = 0.0 (below any real interaction) |

## How soft-label BCE works

Instead of MSE on raw 1–5 ratings, we normalise to [0.2, 1.0] by dividing by 5.  
BCEWithLogitsLoss with soft targets [0, 1] pushes each item's score toward a value
whose sigmoid equals the normalised rating:

- Rating 5 → target 1.0 → score pushed high (most liked)
- Rating 1 → target 0.2 → score pushed slightly positive
- Random negative → target 0.0 → score pushed negative

This means **all interacted items score above all non-interacted items**, regardless of
rating, which directly optimises the HR@10 evaluation metric.  High-rated items also
score higher than low-rated ones, preserving the preference gradient.


## 1. Imports

In [ ]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.sequential import SASRecClassifierEstimator
from skrec.recommender.sequential import SequentialRecommender
from skrec.scorer.sequential import SequentialScorer

# Show training loss logs from the estimator
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/sasrec-ratings")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

## 2. Download MovieLens 1M

Data is stored in `examples/data/ml-1m/` (excluded from git via `.gitignore`).  
If the files already exist from running notebook 1, this step is a no-op.

In [ ]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

## 3. Load and Preprocess — All Ratings as Soft Labels

Unlike the positives-only notebook, we keep **all 1M interactions**.  
OUTCOME is normalised to [0.2, 1.0] by dividing the raw rating by 5, so every real
interaction has a target **strictly above** the random-negative target of 0.0.  
Higher-rated items simply receive a stronger positive gradient.


In [ ]:
# ratings.dat: UserID::MovieID::Rating::Timestamp
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# movies.dat: MovieID::Title::Genres
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

print(f"Ratings: {len(ratings):,}  |  Users: {ratings.UserID.nunique():,}  |  Movies: {ratings.MovieID.nunique():,}")
ratings.head()

In [ ]:
# Map to library column names.
# OUTCOME: normalised rating (1.0–5.0) / 5 → soft label [0.2, 1.0].
# Random negatives sampled at training time will receive target=0.0,
# which is 1 unit below the worst real rating (1.0), giving the model a
# clear learned boundary: unseen < hated < ... < loved.
interactions = pd.DataFrame(
    {
        "USER_ID": ratings["UserID"].astype(str),
        "ITEM_ID": ratings["MovieID"].astype(str),
        "OUTCOME": ratings["Rating"].astype(float) / 5.0,  # 0.2, 0.4, 0.6, 0.8, or 1.0
        # Keep TIMESTAMP as int64 (not str) so sort is numeric, not lexicographic.
        # ML-1M timestamps span 9- and 10-digit values; string sort would order them wrong.
        "TIMESTAMP": ratings["Timestamp"],
    }
)

items = pd.DataFrame({"ITEM_ID": movies["MovieID"].astype(str)})

print(f"Total interactions: {len(interactions):,}")
interactions.head()

## 4. Rating Distribution

Unlike the positives-only notebook, our training sequences now contain the full
distribution of ratings. This section gives a feel for what the model will learn from.

In [ ]:
rating_counts = ratings["Rating"].value_counts().sort_index()
print("Rating distribution (all interactions):")
print("=" * 40)
for rating, count in rating_counts.items():
    bar = "#" * (count // 10000)
    print(f"  {rating} stars: {count:>7,}  ({count / len(ratings):.1%})  {bar}")
print("=" * 40)
print(f"  Mean rating : {ratings['Rating'].mean():.2f}")
print(f"  Median      : {ratings['Rating'].median():.1f}")

## 5. Train / Test Split (Leave-Last-Out on All Interactions)

For each user, the interaction with the largest timestamp is the test item —
regardless of its rating. This is a stricter protocol than the positives-only
notebook: the model must predict the user's *next actual behavior*, not just
their next liked item.

Users with fewer than 5 total interactions are excluded.

In [ ]:
# Sort by timestamp
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

# Keep users with >= 5 interactions
user_counts = interactions.groupby("USER_ID").size()
valid_users = user_counts[user_counts >= 5].index
interactions = interactions[interactions["USER_ID"].isin(valid_users)].reset_index(drop=True)

# Rank items per user (0 = most recent = test)
interactions["rank"] = interactions.groupby("USER_ID").cumcount(ascending=False)
test_df = interactions[interactions["rank"] == 0].drop(columns=["rank"]).reset_index(drop=True)
valid_df = interactions[interactions["rank"] == 1].drop(columns=["rank"]).reset_index(drop=True)

# Training uses ALL interactions (matches original SASRec paper).
# The test item is used as the last training target: the model trains to predict it
# from the preceding history, which is exactly the task being evaluated.
train_df = interactions.drop(columns=["rank"]).reset_index(drop=True)

# Evaluation history: all interactions EXCEPT the test item.
all_except_test_df = interactions[interactions["rank"] >= 1].drop(columns=["rank"]).reset_index(drop=True)

print(f"Train interactions : {len(train_df):,}  (ALL interactions — test item used as last target)")
print(f"Valid interactions : {len(valid_df):,}  (one per user, for reference)")
print(f"Test  interactions : {len(test_df):,}  (one per user)")
print(f"Users              : {train_df.USER_ID.nunique():,}")
print()
print("Rating distribution of held-out test items:")
print(test_df["OUTCOME"].value_counts().sort_index().to_string())

## 6. Save CSVs and Create Datasets

In [ ]:
train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

# Train on ALL interactions (reference SASRec: test item is last training target per user)
if not Path(train_path).exists():
    train_df.to_csv(train_path, index=False)
if not Path(items_path).exists():
    items.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds = ItemsDataset(data_location=items_path)

print(f"Training data: {len(train_df):,} interactions")
print("Datasets created.")

## 7. Build and Train SASRec (Classifier with Soft Labels)

We use `SASRecClassifierEstimator` (BCE) with soft labels and `num_negatives=1`.

**Why soft-label BCE?**  
BCEWithLogitsLoss natively supports targets in [0, 1].  Normalising ratings to
Rating/5 gives a graduated signal while keeping the loss bounded and well-conditioned.  
The model converges faster and learns more discriminative item embeddings than MSE on
raw 1–5 targets.

**Why `num_negatives=1`?**  
At each training step, one random unseen item is sampled with `target=0.0`.
Since the minimum normalised rating is 0.2, this creates a 0.2-unit gap that pushes
unseen items below even disliked ones — the core of the full-signal approach.


In [ ]:
estimator = SASRecClassifierEstimator(
    hidden_units=50,
    num_blocks=2,
    num_heads=1,
    dropout_rate=0.2,
    num_negatives=1,  # original paper: 1 negative per step
    learning_rate=0.001,
    epochs=200,
    batch_size=128,
    optimizer_name="adam",
    loss_fn_name="bce",
    verbose=1,
)

scorer = SequentialScorer(estimator)
recommender = SequentialRecommender(scorer, max_len=200)

print("Training SASRec (soft-label BCE)...")
recommender.train(items_ds=items_ds, interactions_ds=interactions_ds)
print("Training complete.")

## 8. Evaluate: HR@10 and NDCG@10

Same leave-last-out evaluation as notebook 1. The held-out item may have any
rating (1–5) — we check only whether it appears in the top-10, not whether it
was liked. This tests how well the model predicts the user's *next action*.

User order is anchored to `_build_sequences` output to guarantee row alignment
with `recommend()`.

In [ ]:
rng = np.random.default_rng(42)
all_item_ids = np.array(list(scorer.item_names))

known_items = set(scorer.item_names)
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_users = set(eval_test_df["USER_ID"])

# Evaluation history: all interactions EXCEPT the test item.
# The model's last-position representation was trained to predict the test item
# from exactly this context.
eval_history_df = all_except_test_df[all_except_test_df["USER_ID"].isin(eval_users)].copy()
eval_history_df = eval_history_df.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

print(f"Evaluating {len(eval_users):,} users (sampled ranking: 1 positive + 100 negatives)...")

sequences_df = recommender._build_sequences(eval_history_df)
user_order = sequences_df["USER_ID"].tolist()

# Get all-item scores once: (num_users, num_items)
all_scores = recommender.scorer.estimator.predict_proba_with_embeddings(interactions=sequences_df)
item_name_to_idx = {name: i for i, name in enumerate(scorer.item_names)}

gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_items = interactions.groupby("USER_ID")["ITEM_ID"].apply(set).to_dict()

TOP_K = 10
N_NEGATIVES = 100

hits, ndcgs = [], []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None:
        continue

    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)

    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = all_scores[i, candidate_idxs]

    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1

    if rank <= TOP_K:
        hits.append(1)
        ndcgs.append(1.0 / np.log2(rank + 1))
    else:
        hits.append(0)
        ndcgs.append(0.0)

print(f"\n{'=' * 40}")
print(f"Evaluation: 1 positive + {N_NEGATIVES} random negatives")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users evaluated: {len(hits):,}")
print(f"{'=' * 40}")

## 9. Breakdown: HR@10 by Test Item Rating

Since the test item has a known rating, we can check whether the model is
better at predicting items the user *liked* versus items they *disliked*.
A well-trained model should show higher HR@10 for 4–5 star test items because
those items are more similar to the items already in the user's high-rated sequence.

In [ ]:
gt_rating_lookup = eval_test_df.set_index("USER_ID")["OUTCOME"].to_dict()

records = []
for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    test_rating = gt_rating_lookup.get(user_id)
    if test_item is None:
        continue
    seen = user_items.get(user_id, set())
    candidates = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(candidates, size=min(N_NEGATIVES, len(candidates)), replace=False)
    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = all_scores[i, candidate_idxs]
    test_score = all_scores[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1
    hit = int(rank <= TOP_K)
    ndcg = (1.0 / np.log2(rank + 1)) if hit else 0.0
    records.append({"test_rating": int(test_rating), "hit": hit, "ndcg": ndcg})

breakdown = (
    pd.DataFrame(records)
    .groupby("test_rating")
    .agg(
        n_users=("hit", "count"),
        HR10=("hit", "mean"),
        NDCG10=("ndcg", "mean"),
    )
    .round(4)
)

print("HR@10 and NDCG@10 broken down by test item rating (sampled eval):")
print(breakdown.to_string())

## 10. Sample Recommendations

Show top-10 recommendations for a few users alongside their held-out test item
and its rating.

In [ ]:
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()
gt_rating_lookup = eval_test_df.set_index("USER_ID")["OUTCOME"].to_dict()

# Use all-except-test history (same as evaluation) so sequences are consistent
top_k_recs = recommender.recommend(interactions=eval_history_df, top_k=TOP_K)

for user_id in user_order[:5]:
    idx = user_order.index(user_id)
    recs = list(top_k_recs[idx])
    test_item = gt_lookup.get(user_id, "?")
    test_rating = gt_rating_lookup.get(user_id, "?")
    hit = "HIT" if test_item in recs else "MISS"
    print(
        f"\nUser {user_id}  |  Test item: {movie_title.get(test_item, test_item)} (rated {test_rating:.0f}/5)  [{hit}]"
    )
    print("  Top-10 (full-item ranking):")
    for rank, item_id in enumerate(recs, 1):
        marker = " <-- TEST ITEM" if item_id == test_item else ""
        print(f"    {rank:2}. {movie_title.get(item_id, item_id)}{marker}")